In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# ── SLV Model Parameters ──
S0       = 100.0
sigma0   = 0.15
Y0       = 0.0
rho      = -0.50
gamma    = 0.50
kappa_mr = 1.0
T        = 1.0

# ── Simulation Grid ──
n_steps  = 100          # SLV simulation: fine grid (dt = 1/100)
dt       = T / n_steps
t_fine   = np.linspace(0, T, n_steps + 1)

# ── Exercise Dates (monthly, 12 dates) ──
n_exercise = 12
exercise_indices = np.round(np.linspace(0, n_steps, n_exercise + 1)).astype(int)
ts_exercise = t_fine[exercise_indices]   # shape (13,): t0, t1, ..., t12

# ── Calibration / Pricing Settings ──
N_cal     = 10_000      # paths for particle method
N_price   = 100_000     # paths for pricing
sigma_mkt = 0.15        # flat market implied vol
kappa_bw  = 1.0         # bandwidth tuning parameter

K = 100.0               # strike for American put
r = 0.0                 # risk-free rate (SLV notebook uses 0)
q = 0.0                 # dividend yield

In [5]:
# ── Black-Scholes Pricer ──
def blackscholes_price(K, T, S, vol, r=0, q=0, callput='call'):
    F = S * np.exp((r - q) * T)
    w = vol**2 * T
    d1 = (np.log(F / K) + 0.5 * w) / np.sqrt(w)
    d2 = d1 - np.sqrt(w)
    opttype = 1 if callput.lower() == 'call' else -1
    price = opttype * (F * norm.cdf(opttype * d1) - K * norm.cdf(opttype * d2)) * np.exp(-r * T)
    return price

# ── Black-Scholes Implied Vol (scalar) ──
def blackscholes_impv_scalar(K, T, S, value, r=0, q=0, callput='call', tol=1e-6, maxiter=500):
    if (K <= 0) or (T <= 0):
        return np.nan
    F = S * np.exp((r - q) * T)
    K_norm = K / F
    value = value * np.exp(r * T) / F
    opttype = 1 if callput.lower() == 'call' else -1
    value -= max(opttype * (1 - K_norm), 0)
    if value < 0:
        return np.nan
    if value == 0:
        return 0
    j = 1
    p = np.log(K_norm)
    if K_norm >= 1:
        x0 = np.sqrt(2 * p)
        x1 = x0 - (0.5 - K_norm * norm.cdf(-x0) - value) * np.sqrt(2 * np.pi)
        while (abs(x0 - x1) > tol * np.sqrt(T)) and (j < maxiter):
            x0 = x1
            d1 = -p / x1 + 0.5 * x1
            x1 = x1 - (norm.cdf(d1) - K_norm * norm.cdf(d1 - x1) - value) * np.sqrt(2 * np.pi) * np.exp(0.5 * d1**2)
            j += 1
    else:
        x0 = np.sqrt(-2 * p)
        x1 = x0 - (0.5 * K_norm - norm.cdf(-x0) - value) * np.sqrt(2 * np.pi) / K_norm
        while (abs(x0 - x1) > tol * np.sqrt(T)) and (j < maxiter):
            x0 = x1
            d1 = -p / x1 + 0.5 * x1
            x1 = x1 - (K_norm * norm.cdf(x1 - d1) - norm.cdf(-d1) - value) * np.sqrt(2 * np.pi) * np.exp(0.5 * d1**2)
            j += 1
    return x1 / np.sqrt(T)

blackscholes_impv = np.vectorize(blackscholes_impv_scalar, excluded={'callput', 'tol', 'maxiter'})

# ── Quartic Kernel ──
def quartic_kernel(x):
    x = np.clip(x, -1, 1)
    return (x + 1)**2 * (1 - x)**2

In [6]:
# ── Helper Function Tests ──

# TEST 1: blackscholes_price (docstring example)
put_price = blackscholes_price(95, 0.25, 100, 0.2, r=0.05, callput='put')
print(f"BS Put Price: {put_price:.16f}")
print(f"Expected:     1.5342604771222823")
print(f"Match: {np.isclose(put_price, 1.5342604771222823)}\n")

# Put-call parity
call_p = blackscholes_price(100, 1.0, 100, 0.2, r=0.05, callput='call')
put_p  = blackscholes_price(100, 1.0, 100, 0.2, r=0.05, callput='put')
print(f"C - P = {call_p - put_p:.6f}")
print(f"S - Ke^(-rT) = {100 - 100*np.exp(-0.05):.6f}\n")

# TEST 2: blackscholes_impv roundtrip
for v in [0.10, 0.15, 0.20, 0.30]:
    cp = blackscholes_price(100, 1.0, 100, v, callput='call')
    iv = blackscholes_impv(100, 1.0, 100, cp, callput='call')
    print(f"Input vol={v:.2f} → price={cp:.6f} → recovered vol={iv:.6f} → error={abs(iv-v):.2e}")

# TEST 3: quartic_kernel
print(f"\nK(-1)={quartic_kernel(np.array([-1.0]))[0]:.4f}, K(0)={quartic_kernel(np.array([0.0]))[0]:.4f}, K(1)={quartic_kernel(np.array([1.0]))[0]:.4f}")
print(f"Symmetric: K(0.3)={quartic_kernel(np.array([0.3]))[0]:.6f}, K(-0.3)={quartic_kernel(np.array([-0.3]))[0]:.6f}")

BS Put Price: 1.5342604771222823
Expected:     1.5342604771222823
Match: True

C - P = 4.877058
S - Ke^(-rT) = 4.877058

Input vol=0.10 → price=3.987761 → recovered vol=0.100000 → error=8.33e-17
Input vol=0.15 → price=5.978529 → recovered vol=0.150000 → error=2.78e-16
Input vol=0.20 → price=7.965567 → recovered vol=0.200000 → error=2.78e-16
Input vol=0.30 → price=11.923538 → recovered vol=0.300000 → error=3.89e-16

K(-1)=0.0000, K(0)=1.0000, K(1)=0.0000
Symmetric: K(0.3)=0.828100, K(-0.3)=0.828100


In [7]:
def quartic_kernel(x):
    x = np.clip(x, -1, 1)
    return (x + 1)**2 * (1 - x)**2


def compute_leverage_on_grid(S_particles, Y_particles, S_grid_k,
                              t_k, sigma0, sigma_mkt, S0, kappa_bw, N_cal):
    with np.errstate(invalid='ignore', divide='ignore'):
        h = kappa_bw * sigma_mkt * S0 * np.sqrt(max(t_k, 0.15)) * N_cal**(-0.2)
        u = (S_particles[None, :] - S_grid_k[:, None]) / h
        K_vals = quartic_kernel(u)
        stoch_var = sigma0**2 * np.exp(2 * Y_particles)
        numer = K_vals @ stoch_var
        denom = K_vals.sum(axis=1)
        cond_exp = np.where(denom > 0, numer / denom, sigma0**2)
        lev_grid = np.sqrt(np.clip(sigma_mkt**2 / cond_exp, 1e-4, 1e2))
    return lev_grid


def particle_method(S0, sigma0, Y0, rho, gamma, kappa_mr, T,
                    n_steps, N_cal, sigma_mkt, kappa_bw, n_grid=200, seed=42):
    dt_sim    = T / n_steps
    exp_kdt   = np.exp(-kappa_mr * dt_sim)
    std_Y     = gamma * np.sqrt((1 - np.exp(-2 * kappa_mr * dt_sim)) / (2 * kappa_mr))
    rho_bar   = rho * np.sqrt(2 * (1 - exp_kdt) / (kappa_mr * dt_sim * (1 + exp_kdt)))
    rng       = np.random.default_rng(seed)
    logS      = np.full(N_cal, np.log(S0))
    Y         = np.full(N_cal, float(Y0))
    lev_store = []
    for k in range(n_steps):
        t_k         = k * dt_sim
        S_particles = np.exp(logS)
        Y_old       = Y.copy()
        S_lo = S_particles.min(); S_hi = S_particles.max()
        margin   = 0.05 * (S_hi - S_lo + 1e-8)
        S_grid_k = np.linspace(S_lo - margin, S_hi + margin, n_grid)
        lev_grid = compute_leverage_on_grid(S_particles, Y_old, S_grid_k,
                                             t_k, sigma0, sigma_mkt, S0, kappa_bw, N_cal)
        lev_store.append((S_grid_k.copy(), lev_grid.copy()))
        lev_p = np.interp(S_particles, S_grid_k, lev_grid)
        Z1 = rng.standard_normal(N_cal); Z2 = rng.standard_normal(N_cal)
        Y    = exp_kdt * Y_old + std_Y * Z2
        veff = sigma0 * np.exp(Y_old) * lev_p
        logS = (logS - 0.5 * veff**2 * dt_sim
                + veff * np.sqrt(dt_sim) * (np.sqrt(1 - rho_bar**2) * Z1 + rho_bar * Z2))
    return lev_store, logS, Y


def slv_paths_matrix(S0, sigma0, Y0, rho, gamma, kappa_mr, T,
                     n_steps, N, lev_store, exercise_indices, seed=0):
    dt_sim  = T / n_steps
    exp_kdt = np.exp(-kappa_mr * dt_sim)
    std_Y   = gamma * np.sqrt((1 - np.exp(-2 * kappa_mr * dt_sim)) / (2 * kappa_mr))
    rho_bar = rho * np.sqrt(2 * (1 - exp_kdt) / (kappa_mr * dt_sim * (1 + exp_kdt)))
    rng     = np.random.default_rng(seed)
    logS    = np.full(N, np.log(S0))
    Y       = np.full(N, float(Y0))
    ex_set  = set(exercise_indices[1:])
    paths   = np.empty((len(exercise_indices), N))
    paths[0] = S0
    ex_idx  = 1
    for k in range(n_steps):
        S_grid_k, lev_grid_k = lev_store[k]
        S_now = np.exp(logS)
        Y_old = Y.copy()
        lev   = np.interp(S_now, S_grid_k, lev_grid_k)
        Z1 = rng.standard_normal(N); Z2 = rng.standard_normal(N)
        Y    = exp_kdt * Y_old + std_Y * Z2
        veff = sigma0 * np.exp(Y_old) * lev
        logS = (logS - 0.5 * veff**2 * dt_sim
                + veff * np.sqrt(dt_sim) * (np.sqrt(1 - rho_bar**2) * Z1 + rho_bar * Z2))
        if (k + 1) in ex_set:
            paths[ex_idx] = np.exp(logS)
            ex_idx += 1
    return paths


def regression_polynomial(X, Y, S_current, degree=3, **kwargs):
    coeffs = np.polyfit(X, Y, deg=degree)
    return np.polyval(coeffs, S_current), {'type': 'polynomial', 'coeffs': coeffs, 'degree': degree}


def ls_pricer(paths, K, r, ts, regression_func, **regression_params):
    n_steps, n_paths = paths.shape
    payoff = np.maximum(K - paths[-1], 0)
    exercise_policy = {'models': [], 'ts': ts, 'K': K, 'r': r}
    for i in range(n_steps - 2, 0, -1):
        discount         = np.exp(-r * (ts[i + 1] - ts[i]))
        payoff_discounted = payoff * discount
        exercise_value   = np.maximum(K - paths[i], 0)
        X = paths[i]; Y_reg = payoff_discounted
        tau = ts[-1] - ts[i]
        cont_value, model = regression_func(X, Y_reg, X, K=K, r=r, tau=tau, **regression_params)
        exercise_policy['models'].insert(0, model)
        exercise_mask = exercise_value > cont_value
        payoff_new = payoff_discounted.copy()
        payoff_new[exercise_mask] = exercise_value[exercise_mask]
        payoff = payoff_new
    price = np.mean(payoff * np.exp(-r * (ts[1] - ts[0])))
    return price, exercise_policy


def exer_or_cont(i, S, models, K, r, ts):
    S = np.atleast_1d(S)
    if i >= len(ts) - 1:
        return np.maximum(K - S, 0) > 0
    if i < 1 or i - 1 >= len(models):
        return np.zeros(len(S), dtype=bool)
    model = models[i - 1]
    exercise_value = np.maximum(K - S, 0)
    cont_value = np.polyval(model['coeffs'], S)
    return exercise_value > cont_value


def nested_mc(S, vol, r, q, i, ts, nnested, models, K):
    nested_paths = np.full(nnested, S, dtype=float)
    tot_payoff = 0
    for j in range(i + 1, len(ts)):
        dt = ts[j] - ts[j - 1]
        dW = np.random.randn(len(nested_paths)) * np.sqrt(dt)
        nested_paths = nested_paths * np.exp((r - q) * dt) * np.exp(-0.5 * vol**2 * dt + vol * dW)
        exer_vals = np.maximum(K - nested_paths, 0)
        if j < len(ts) - 1:
            ind = exer_or_cont(j, nested_paths, models, K, r, ts)
            tot_payoff += np.sum(exer_vals[ind]) * np.exp(-r * ts[j])
            nested_paths = nested_paths[~ind]
            if len(nested_paths) == 0:
                break
        else:
            tot_payoff += np.sum(exer_vals) * np.exp(-r * ts[j])
    return tot_payoff / nnested


lev_store, _, _ = particle_method(S0, sigma0, Y0, rho, gamma, kappa_mr, T,
                                   n_steps, N_cal, sigma_mkt, kappa_bw)

paths_train = slv_paths_matrix(S0, sigma0, Y0, rho, gamma, kappa_mr, T,
                                n_steps, N_cal, lev_store, exercise_indices, seed=42)

price_lsm_slv, policy_slv = ls_pricer(paths_train, K, r, ts_exercise,
                                        regression_polynomial, degree=3)
models_slv = policy_slv['models']

N_ba     = 500
N_nested = 1000
paths_ba = slv_paths_matrix(S0, sigma0, Y0, rho, gamma, kappa_mr, T,
                             n_steps, N_ba, lev_store, exercise_indices, seed=777)

exer_slv = lambda S: np.maximum(K - S, 0)

V  = np.full(paths_ba.shape, np.nan)
EV = np.full(paths_ba.shape, np.nan)
V[0] = EV[0] = price_lsm_slv

for i in range(1, len(ts_exercise) - 1):
    ev  = exer_slv(paths_ba[i])
    ind = exer_or_cont(i, paths_ba[i], models_slv, K, r, ts_exercise)
    for j in np.nonzero(ind)[0]:
        V[i, j]  = ev[j] * np.exp(-r * ts_exercise[i])
        EV[i, j] = nested_mc(paths_ba[i, j], sigma_mkt, r, q, i,
                              ts_exercise, N_nested, models_slv, K)
    for j in np.nonzero(~ind)[0]:
        V[i, j]  = nested_mc(paths_ba[i, j], sigma_mkt, r, q, i,
                              ts_exercise, N_nested, models_slv, K)
        EV[i, j] = V[i, j]

V[-1]      = exer_slv(paths_ba[-1]) * np.exp(-r * ts_exercise[-1])
hedges     = np.zeros(paths_ba.shape)
hedges[1:] = np.cumsum(V[1:] - EV[:-1], axis=0)

upper_slv = np.mean(np.amax(
    exer_slv(paths_ba[1:]) * np.exp(-r * ts_exercise[1:, np.newaxis]) - hedges[1:],
    axis=0
))

print(f"SLV American Put  (K={K}, T={T}, r={r})")
print(f"  LSM Lower : {price_lsm_slv:.4f}")
print(f"  BA  Upper : {upper_slv:.4f}")

SLV American Put  (K=100.0, T=1.0, r=0.0)
  LSM Lower : 5.8359
  BA  Upper : 6.0053


In [8]:
K_list = np.array([70, 80, 90, 100, 110, 120, 130, 140], dtype=float)

def _impv(K_, price, callput='call'):
    from scipy.optimize import brentq
    def obj(v):
        sqT = np.sqrt(T)
        d1 = np.log(S0/K_)/(v*sqT) + 0.5*v*sqT
        d2 = d1 - v*sqT
        if callput == 'call':
            return S0*norm.cdf(d1) - K_*norm.cdf(d2) - price
        else:
            return K_*norm.cdf(-d2) - S0*norm.cdf(-d1) - price
    try:
        return brentq(obj, 1e-6, 5.0)
    except:
        return np.nan

def bs_european_put(K_, T_, S_, v_):
    sqT = np.sqrt(T_)
    d1  = np.log(S_/K_)/(v_*sqT) + 0.5*v_*sqT
    d2  = d1 - v_*sqT
    return K_*norm.cdf(-d2) - S_*norm.cdf(-d1)

print("=" * 60)
print("1. quartic_kernel")
print("=" * 60)
assert quartic_kernel(np.array([-1.0]))[0] == 0.0
assert quartic_kernel(np.array([ 1.0]))[0] == 0.0
assert quartic_kernel(np.array([ 0.0]))[0] == 1.0
assert np.isclose(quartic_kernel(np.array([0.3]))[0],
                  quartic_kernel(np.array([-0.3]))[0])
print("  PASS — boundary, peak, symmetry")

print()
print("=" * 60)
print("2. particle_method — smile calibration check")
print("=" * 60)
_lev_store, _logS_cal, _ = particle_method(S0, sigma0, Y0, rho, gamma, kappa_mr, T,
                                             n_steps, N_cal, sigma_mkt, kappa_bw)
S_T_check = np.exp(_logS_cal)
call_p = np.array([np.maximum(S_T_check - K_, 0).mean() for K_ in K_list])
put_p  = np.array([np.maximum(K_ - S_T_check, 0).mean() for K_ in K_list])
ivs = np.where(K_list < S0,
               [_impv(K_, p, 'put')  for K_, p in zip(K_list, put_p)],
               [_impv(K_, p, 'call') for K_, p in zip(K_list, call_p)])
max_err = np.nanmax(np.abs(ivs - sigma_mkt))
print(f"  {'Strike':>8}  {'IV':>8}  {'Error':>10}")
for K_, iv in zip(K_list, ivs):
    print(f"  {K_:>8.0f}  {iv:>8.4f}  {abs(iv-sigma_mkt):>10.4f}")
_cal_pass = max_err < 0.01
print(f"  Max error: {max_err:.4f}  {'PASS' if _cal_pass else 'FAIL'}")

print()
print("=" * 60)
print("3. slv_paths_matrix — shape & basic sanity")
print("=" * 60)
_paths = slv_paths_matrix(S0, sigma0, Y0, rho, gamma, kappa_mr, T,
                           n_steps, 5000, _lev_store, exercise_indices, seed=0)
assert _paths.shape == (len(exercise_indices), 5000)
assert np.all(_paths[0] == S0)
assert not np.any(np.isnan(_paths))
assert np.all(_paths > 0)
_mean_err = abs(_paths[-1].mean() - S0)
print(f"  shape : {_paths.shape}  PASS")
print(f"  t=0   : all == {S0}  PASS")
print(f"  NaN   : none  PASS")
print(f"  S_T   : mean={_paths[-1].mean():.2f}  std={_paths[-1].std():.2f}")
_path_pass = _mean_err < 3.0
print(f"  E[S_T] error vs S0: {_mean_err:.4f}  {'PASS' if _path_pass else 'FAIL'}")

print()
print("=" * 60)
print("4. ls_pricer — lower bound sanity (r=0: American ≈ European)")
print("=" * 60)
_price_lsm, _policy = ls_pricer(_paths, K, r, ts_exercise, regression_polynomial, degree=3)
_bs_eur = bs_european_put(K, T, S0, sigma_mkt)
_premium = _price_lsm - _bs_eur
print(f"  LSM Lower  : {_price_lsm:.4f}")
print(f"  BS European: {_bs_eur:.4f}")
print(f"  Difference : {_premium:.4f}")
print(f"  Note: r=0 so early exercise is suboptimal → |diff| driven by MC noise")
_lsm_pass = abs(_premium) < 0.5
print(f"  |diff| < 0.5 : {'PASS' if _lsm_pass else 'FAIL'}")
assert len(_policy['models']) == len(ts_exercise) - 2
print(f"  # models   : {len(_policy['models'])} (expected {len(ts_exercise)-2})  PASS")

print()
print("=" * 60)
print("5. exer_or_cont — boundary conditions")
print("=" * 60)
_models = _policy['models']
_deep_otm = np.array([150.0, 160.0])
_exer_otm = exer_or_cont(1, _deep_otm, _models, K, r, ts_exercise)
_exer_mat_itm = exer_or_cont(len(ts_exercise)-1, np.array([50.0, 60.0]), _models, K, r, ts_exercise)
_exer_mat_otm = exer_or_cont(len(ts_exercise)-1, np.array([150.0, 160.0]), _models, K, r, ts_exercise)
_otm_pass = not any(_exer_otm)
_mat_itm_pass = all(_exer_mat_itm)
_mat_otm_pass = not any(_exer_mat_otm)
print(f"  Deep OTM at t=1 → no exercise  : {_exer_otm}  {'PASS' if _otm_pass else 'FAIL'}")
print(f"  At maturity ITM → exercise      : {_exer_mat_itm}  {'PASS' if _mat_itm_pass else 'FAIL'}")
print(f"  At maturity OTM → no exercise   : {_exer_mat_otm}  {'PASS' if _mat_otm_pass else 'FAIL'}")

print()
print("=" * 60)
print("6. BA upper bound — valid sandwich")
print("=" * 60)
print(f"  LSM Lower : {price_lsm_slv:.4f}")
print(f"  BA  Upper : {upper_slv:.4f}")
print(f"  Gap       : {upper_slv - price_lsm_slv:.4f}")
_ba_pass = price_lsm_slv < upper_slv
print(f"  Lower < Upper : {'PASS' if _ba_pass else 'FAIL'}")

print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  quartic_kernel     : PASS")
print(f"  particle_method    : {'PASS' if _cal_pass else 'FAIL'}")
print(f"  slv_paths_matrix   : {'PASS' if _path_pass else 'FAIL'}")
print(f"  ls_pricer          : {'PASS' if _lsm_pass else 'FAIL'}")
print(f"  exer_or_cont       : {'PASS' if _otm_pass and _mat_itm_pass and _mat_otm_pass else 'FAIL'}")
print(f"  BA upper bound     : {'PASS' if _ba_pass else 'FAIL'}")

1. quartic_kernel
  PASS — boundary, peak, symmetry

2. particle_method — smile calibration check
    Strike        IV       Error
        70    0.1497      0.0003
        80    0.1503      0.0003
        90    0.1496      0.0004
       100    0.1506      0.0006
       110    0.1517      0.0017
       120    0.1527      0.0027
       130    0.1532      0.0032
       140    0.1552      0.0052
  Max error: 0.0052  PASS

3. slv_paths_matrix — shape & basic sanity
  shape : (13, 5000)  PASS
  t=0   : all == 100.0  PASS
  NaN   : none  PASS
  S_T   : mean=100.30  std=14.97
  E[S_T] error vs S0: 0.3039  PASS

4. ls_pricer — lower bound sanity (r=0: American ≈ European)
  LSM Lower  : 5.6722
  BS European: 5.9785
  Difference : -0.3063
  Note: r=0 so early exercise is suboptimal → |diff| driven by MC noise
  |diff| < 0.5 : PASS
  # models   : 11 (expected 11)  PASS

5. exer_or_cont — boundary conditions
  Deep OTM at t=1 → no exercise  : [False False]  PASS
  At maturity ITM → exercise      :